In [ ]:
# PAD-UFES-20 Ablation — Config 3: Image + Metadata (ConvNeXt-Tiny)

import os, glob, random, warnings, logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import cv2
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from torchinfo import summary
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

In [ ]:
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32

train_tfms = transforms.Compose([
    transforms.Resize(288),
    transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2, hue=0.05)], p=0.5),
    transforms.RandomAffine(degrees=180, translate=(0.1,0.1), scale=(0.9,1.1), shear=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_test_tfms = transforms.Compose([
    transforms.Resize(288),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
meta_path = "/kaggle/input/datasets/sujansarkar/pad-ufes-20-classification/metadata.csv"
df        = pd.read_csv(meta_path)

gss_train = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss_train.split(df, groups=df["patient_id"]))
train_df = df.iloc[train_idx].copy()
temp_df  = df.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df["patient_id"]))
val_df  = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Train:", dict(train_df["diagnostic"].value_counts()))
print("Val:  ", dict(val_df["diagnostic"].value_counts()))
print("Test: ", dict(test_df["diagnostic"].value_counts()))

In [ ]:
def crop_and_save_dataset(input_dir, output_dir, padding=20, min_area_ratio=0.05):
    os.makedirs(output_dir, exist_ok=True)
    image_paths = glob.glob(os.path.join(input_dir, "**", "*.png"), recursive=True)
    for img_path in tqdm(image_paths, desc="Auto-Cropping"):
        rel_path = os.path.relpath(img_path, input_dir)
        out_path = os.path.join(output_dir, rel_path)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        img = cv2.imread(img_path)
        if img is None: continue
        img_area = img.shape[0] * img.shape[1]
        gray   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur   = cv2.GaussianBlur(gray, (15,15), 0)
        _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        kernel  = np.ones((9,9), np.uint8)
        closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: cv2.imwrite(out_path, img); continue
        lc = max(contours, key=cv2.contourArea)
        ca = cv2.contourArea(lc)
        if ca < img_area*min_area_ratio or ca > img_area*0.95:
            cv2.imwrite(out_path, img); continue
        x,y,w,h = cv2.boundingRect(lc)
        x=max(0,x-padding); y=max(0,y-padding)
        w=min(img.shape[1]-x,w+2*padding); h=min(img.shape[0]-y,h+2*padding)
        cv2.imwrite(out_path, img[y:y+h, x:x+w])

ORIGINAL_DATA = "/kaggle/input/datasets/sujansarkar/pad-ufes-20-classification/images"
CROPPED_DATA  = "/kaggle/working/cropped_pad_ufes_20"

if not os.path.exists(CROPPED_DATA):
    crop_and_save_dataset(ORIGINAL_DATA, CROPPED_DATA)
else:
    print("Cropped dataset already exists — skipping.")

In [ ]:
meta_path = "/kaggle/input/datasets/sujansarkar/pad-ufes-20-classification/metadata.csv"
df_meta   = pd.read_csv(meta_path)

# Re-apply the same GroupShuffleSplit to match train/val/test splits exactly
gss_train = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss_train.split(df_meta, groups=df_meta["patient_id"]))
train_df_full = df_meta.iloc[train_idx].copy()
temp_df_full  = df_meta.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df_full, groups=temp_df_full["patient_id"]))
val_df_full  = temp_df_full.iloc[val_idx].copy()
test_df_full = temp_df_full.iloc[test_idx].copy()

for split in [train_df_full, val_df_full, test_df_full]:
    split.set_index("img_id", inplace=True)

# Numerical: age — normalise using train stats only (no data leakage)
AGE_MEAN = train_df_full["age"].mean()
AGE_STD  = train_df_full["age"].std() + 1e-8
for m in [train_df_full, val_df_full, test_df_full]:
    m["age_norm"] = (m["age"].fillna(AGE_MEAN) - AGE_MEAN) / AGE_STD

# Binary symptom flags
BINARY_COLS = ["itch", "grew", "hurt", "changed", "bleed", "elevation"]
for col in BINARY_COLS:
    for m in [train_df_full, val_df_full, test_df_full]:
        m[col] = m[col].apply(lambda x: 1.0 if str(x) == "True" else 0.0)

# Categorical: gender, region — one-hot, fit on train only
for col in ["gender", "region"]:
    for m in [train_df_full, val_df_full, test_df_full]:
        m[col] = m[col].fillna("Unknown")

train_df_full = pd.get_dummies(train_df_full, columns=["gender","region"],
                                dummy_na=False, dtype=float)
for split_name, split in [("val", val_df_full), ("test", test_df_full)]:
    for col in ["gender", "region"]:
        dummies = pd.get_dummies(split[col], prefix=col, dummy_na=False, dtype=float)
        split.drop(columns=[col], inplace=True)
        for c in dummies.columns:
            split[c] = dummies[c]

# Align val/test columns to train (fill missing one-hot columns with 0)
val_df_full  = val_df_full.reindex(columns=train_df_full.columns, fill_value=0)
test_df_full = test_df_full.reindex(columns=train_df_full.columns, fill_value=0)

META_COLS = (["age_norm"] + BINARY_COLS +
             [c for c in train_df_full.columns
              if c.startswith("gender_") or c.startswith("region_")])
NUM_META  = len(META_COLS)
print(f"Tabular features: {NUM_META}")
print(f"  {META_COLS}")

In [ ]:
class ImageMetaDataset(Dataset):
    def __init__(self, df, root_dir, meta_cols, transform=None, class_to_idx=None):
        self.df        = df
        self.meta_cols = meta_cols
        self.transform = transform
        self._zero     = torch.zeros(len(meta_cols))

        self.image_paths = {}
        for path in glob.glob(os.path.join(root_dir, "**", "*.png"), recursive=True):
            self.image_paths[os.path.basename(path)] = path

        if class_to_idx is not None:
            self.class_to_idx = class_to_idx
        else:
            classes = sorted(df["diagnostic"].unique())
            self.class_to_idx = {c: i for i, c in enumerate(classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.index[idx]
        row    = self.df.iloc[idx]
        label  = self.class_to_idx[row["diagnostic"]]

        img_path = self.image_paths.get(str(img_id), None)
        if img_path is None:
            image = torch.zeros(3, 256, 256)
        else:
            image = Image.open(img_path).convert("RGB")
            if self.transform: image = self.transform(image)

        tab = torch.tensor(row[self.meta_cols].values.astype(float), dtype=torch.float32)
        return image, tab, label

# Build class_to_idx from train — shared across splits
classes      = sorted(train_df_full["diagnostic"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

train_dataset = ImageMetaDataset(train_df_full, CROPPED_DATA, META_COLS, train_tfms,    class_to_idx)
val_dataset   = ImageMetaDataset(val_df_full,   CROPPED_DATA, META_COLS, val_test_tfms, class_to_idx)
test_dataset  = ImageMetaDataset(test_df_full,  CROPPED_DATA, META_COLS, val_test_tfms, class_to_idx)

# FocalLoss class weights — inverse sqrt-frequency, normalised to mean=1
train_targets        = [class_to_idx[l] for l in train_df_full["diagnostic"]]
class_counts         = Counter(train_targets)
NUM_CLS              = len(class_counts)
total_samples        = sum(class_counts.values())
raw_weights          = [total_samples / (NUM_CLS * class_counts[i] ** 0.5) for i in range(NUM_CLS)]
mean_w               = sum(raw_weights) / NUM_CLS
class_weights_tensor = torch.FloatTensor([w / mean_w for w in raw_weights])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
IDX_TO_CLASS = {
    class_to_idx["ACK"]: "Actinic Keratosis (ACK)",
    class_to_idx["BCC"]: "Basal Cell Carcinoma (BCC)",
    class_to_idx["MEL"]: "Melanoma (MEL)",
    class_to_idx["NEV"]: "Nevus (NEV)",
    class_to_idx["SCC"]: "Squamous Cell Carcinoma (SCC)",
    class_to_idx["SEK"]: "Seborrheic Keratosis (SEK)",
}
NUM_CLASSES = len(IDX_TO_CLASS)
class_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]
for i, n in enumerate(class_names): print(i, "→", n)

In [ ]:
class ConvNeXtMetaNet(nn.Module):
    def __init__(self, num_meta_features, num_classes=6, pretrained=True, freeze_backbone=False):
        super().__init__()
        convnext = models.convnext_tiny(
            weights=models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        )
        self.convnext_features = convnext.features
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.tabular_mlp = nn.Sequential(
            nn.Linear(num_meta_features, 128), nn.BatchNorm1d(128),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(inplace=True)
        )

        # 768 (ConvNeXt) + 64 (tabular) = 832
        self.classifier = nn.Sequential(
            nn.Linear(768 + 64, 256),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

        if freeze_backbone:
            for p in self.convnext_features.parameters():
                p.requires_grad = False

    def forward(self, x, tab):
        f   = self.pool(self.convnext_features(x)).squeeze(-1).squeeze(-1)
        tab = self.tabular_mlp(tab)
        return self.classifier(torch.cat([f, tab], dim=-1))


model = ConvNeXtMetaNet(
    num_meta_features=NUM_META,
    num_classes=NUM_CLASSES,
    pretrained=True,
    freeze_backbone=True       # frozen for Phase 1
).to(DEVICE)

summary(
    model,
    input_size=[(BATCH_SIZE, 3, 256, 256), (BATCH_SIZE, NUM_META)],
    dtypes=[torch.float32, torch.float32],
    col_names=["input_size", "output_size", "num_params"],
    depth=3
)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        # label_smoothing removed — it softens targets in the opposite direction
        # to focal loss's hard-example focus, partially cancelling the effect
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction="none")

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

In [ ]:
scaler = GradScaler("cuda")

def compute_specificity(y_true, y_pred, n):
    cm = confusion_matrix(y_true, y_pred, labels=range(n))
    specs = []
    for i in range(n):
        TP=cm[i,i]; FP=cm[:,i].sum()-TP; FN=cm[i,:].sum()-TP; TN=cm.sum()-(TP+FP+FN)
        specs.append(TN/(TN+FP) if (TN+FP)>0 else 0)
    return np.mean(specs)

def mixup(images, tab, labels, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0), device=images.device)
    return (lam*images+(1-lam)*images[idx],
            lam*tab+(1-lam)*tab[idx],
            labels, labels[idx], lam)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss=0.0; yt,yp=[],[]
    for images, tab, labels in tqdm(loader, desc="Train", leave=False):
        images=images.to(device); tab=tab.to(device); labels=labels.to(device)
        mix_img,mix_tab,ya,yb,lam = mixup(images, tab, labels)
        optimizer.zero_grad()
        with autocast("cuda"):
            out  = model(mix_img, mix_tab)
            loss = lam*criterion(out,ya)+(1-lam)*criterion(out,yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        running_loss += loss.item()*images.size(0)
        yt.extend(ya.cpu().numpy()); yp.extend(out.argmax(1).cpu().numpy())
    L=running_loss/len(yt)
    return (L, accuracy_score(yt,yp),
            precision_score(yt,yp,average="macro",zero_division=0),
            recall_score(yt,yp,average="macro",zero_division=0),
            f1_score(yt,yp,average="macro",zero_division=0),
            compute_specificity(yt,yp,NUM_CLASSES))

def evaluate(model, loader, criterion, device):
    model.eval(); running_loss=0.0; yt,yp=[],[]
    with torch.no_grad():
        for images, tab, labels in tqdm(loader, desc="Eval", leave=False):
            images=images.to(device); tab=tab.to(device); labels=labels.to(device)
            out=model(images,tab); loss=criterion(out,labels)
            running_loss+=loss.item()*images.size(0)
            yt.extend(labels.cpu().numpy()); yp.extend(out.argmax(1).cpu().numpy())
    L=running_loss/len(yt)
    return (L, accuracy_score(yt,yp),
            precision_score(yt,yp,average="macro",zero_division=0),
            recall_score(yt,yp,average="macro",zero_division=0),
            f1_score(yt,yp,average="macro",zero_division=0),
            compute_specificity(yt,yp,NUM_CLASSES), yt, yp)

In [ ]:
criterion = FocalLoss(weight=class_weights_tensor.to(DEVICE), gamma=2.0)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-2
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

PHASE1_EPOCHS = 20
train_losses, val_losses = [], []
best_val_f1=0.0; best_model_state=None

for epoch in range(PHASE1_EPOCHS):
    print(f"\nPhase 1 Epoch [{epoch+1}/{PHASE1_EPOCHS}]")
    tr = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl = evaluate(model, val_loader, criterion, DEVICE)
    train_losses.append(tr[0]); val_losses.append(vl[0])
    scheduler.step()
    print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")
    print(f"Train | Loss:{tr[0]:.4f} Acc:{tr[1]:.4f} F1:{tr[4]:.4f} Spec:{tr[5]:.4f}")
    print(f"Val   | Loss:{vl[0]:.4f} Acc:{vl[1]:.4f} F1:{vl[4]:.4f} Spec:{vl[5]:.4f}")
    if vl[4] > best_val_f1:
        best_val_f1=vl[4]; best_model_state=model.state_dict()
        print(f"Val F1 improved ({best_val_f1:.4f}) — saving")

if best_model_state: model.load_state_dict(best_model_state)
print(f"Phase 1 done. Best val F1: {best_val_f1:.4f}")
for p in model.parameters(): p.requires_grad = True

In [ ]:
for param in model.parameters():
    param.requires_grad = True

In [ ]:
optimizer = torch.optim.AdamW([
    {"params": model.convnext_features.parameters(), "lr": 1e-5},
    {"params": model.tabular_mlp.parameters(),       "lr": 1e-4},
    {"params": model.classifier.parameters(),        "lr": 1e-4},
], weight_decay=5e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7
)

PHASE2_EPOCHS = 60
fine_train_losses, fine_val_losses = [], []
RESTART_EPOCHS = {10, 30}
patience=15; counter=0

for epoch in range(PHASE2_EPOCHS):
    print(f"\nPhase 2 Epoch [{epoch+1}/{PHASE2_EPOCHS}]")
    if (epoch+1) in RESTART_EPOCHS: counter=0; print("LR restart — patience reset")
    tr = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl = evaluate(model, val_loader, criterion, DEVICE)
    fine_train_losses.append(tr[0]); fine_val_losses.append(vl[0])
    scheduler.step()
    print(f"LR (backbone):{optimizer.param_groups[0]['lr']:.2e} | LR (head):{optimizer.param_groups[-1]['lr']:.2e}")
    print(f"Train | Loss:{tr[0]:.4f} Acc:{tr[1]:.4f} F1:{tr[4]:.4f} Spec:{tr[5]:.4f}")
    print(f"Val   | Loss:{vl[0]:.4f} Acc:{vl[1]:.4f} F1:{vl[4]:.4f} Spec:{vl[5]:.4f}")
    if vl[4] > best_val_f1:
        best_val_f1=vl[4]; best_model_state=model.state_dict()
        torch.save(best_model_state, "/kaggle/working/PAD_C3_ImageMeta_Best.pth")
        counter=0; print(f"Val F1 improved ({best_val_f1:.4f}) — saving")
    else:
        counter+=1
        if counter>=patience: print(f"Early stopping at epoch {epoch+1}"); break

if best_model_state: model.load_state_dict(best_model_state)
print(f"Phase 2 done. Best val F1: {best_val_f1:.4f}")

In [ ]:
def print_report(split, model, loader, criterion):
    L,acc,prec,rec,f1,spec,yt,yp = evaluate(model, loader, criterion, DEVICE)
    print(f"\n{'='*50}\n{split.upper()} REPORT")
    print(f"Loss:{L:.4f} | Acc:{acc:.4f} | Spec:{spec:.4f}")
    print(classification_report(yt, yp, target_names=class_names, zero_division=0, digits=4))
    mprec=precision_score(yt,yp,average="macro",zero_division=0)
    mrec=recall_score(yt,yp,average="macro",zero_division=0)
    mf1=f1_score(yt,yp,average="macro",zero_division=0)
    print(f"Macro → Prec:{mprec:.4f} | Rec:{mrec:.4f} | F1:{mf1:.4f} | Spec:{spec:.4f}")
    return yt, yp

train_yt,train_yp = print_report("Train",      model, train_loader, criterion)
val_yt,val_yp     = print_report("Validation", model, val_loader,   criterion)
test_yt,test_yp   = print_report("Test",       model, test_loader,  criterion)

In [ ]:
from scipy import stats as scipy_stats

CONFIG = "C3"   # "C2" | "C3"

N_BOOTSTRAP = 1000
rng        = np.random.default_rng(42)
y_true_arr = np.array(test_yt)
y_pred_arr = np.array(test_yp)
n_test     = len(y_true_arr)

boot = {"Accuracy": [], "F1 (Macro)": [], "Precision (Macro)": [], "Recall (Macro)": [], "Specificity": []}
per_class_f1 = {name: [] for name in class_names}

def compute_specificity_boot(y_t, y_p, num_classes):
    cm = confusion_matrix(y_t, y_p, labels=list(range(num_classes)))
    specs = []
    for i in range(num_classes):
        TP = cm[i, i]; FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP; TN = cm.sum() - (TP + FP + FN)
        specs.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
    return float(np.mean(specs))

for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_test, size=n_test)
    y_t, y_p = y_true_arr[idx], y_pred_arr[idx]
    boot["Accuracy"].append(accuracy_score(y_t, y_p))
    boot["F1 (Macro)"].append(f1_score(y_t, y_p, average="macro", zero_division=0))
    boot["Precision (Macro)"].append(precision_score(y_t, y_p, average="macro", zero_division=0))
    boot["Recall (Macro)"].append(recall_score(y_t, y_p, average="macro", zero_division=0))
    boot["Specificity"].append(compute_specificity_boot(y_t, y_p, NUM_CLASSES))
    pc = f1_score(y_t, y_p, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    for ci, name in enumerate(class_names):
        per_class_f1[name].append(pc[ci])

def ci95(arr): return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

short_names = [n.split("(")[-1].replace(")", "").strip() if "(" in n else n
               for n in class_names]

W = 70
print("\n" + "=" * W)
print(f"  PAD-UFES-20 — {CONFIG} — Bootstrap CI (n={N_BOOTSTRAP})")
print("=" * W)
print(f"{'Metric':<22} {'Mean':>7}  {'Std':>7}  {'CI Lower':>10}  {'CI Upper':>10}")
print("=" * W)
for metric, values in boot.items():
    arr = np.array(values); lo, hi = ci95(arr)
    print(f"{metric:<22} {arr.mean():>7.4f}  {arr.std():>7.4f}  {lo:>10.4f}  {hi:>10.4f}")
print("-" * W)
print(f"\n{'Class':<12} {'Mean':>7}  {'Std':>7}  {'CI Lower':>10}  {'CI Upper':>10}")
print("-" * W)
for sname, name in zip(short_names, class_names):
    arr = np.array(per_class_f1[name]); lo, hi = ci95(arr)
    print(f"{sname:<12} {arr.mean():>7.4f}  {arr.std():>7.4f}  {lo:>10.4f}  {hi:>10.4f}")
print("=" * W)

fig, axes = plt.subplots(1, 5, figsize=(18, 6))
fig.subplots_adjust(top=0.82, wspace=0.4)
for ax, (label, values) in zip(axes, boot.items()):
    arr = np.array(values); lo, hi = ci95(arr); mn, sd = arr.mean(), arr.std()
    ax.boxplot(arr, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor="#4c72b0", alpha=0.75),
               medianprops=dict(color="white", linewidth=2),
               whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
               flierprops=dict(marker="o", markersize=3, markerfacecolor="gray", alpha=0.5))
    ax.set_title(label, fontsize=10, fontweight="bold", pad=8)
    ax.set_xticks([]); ax.set_ylabel("Score", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.text(1, ax.get_ylim()[1], f"μ={mn:.4f}\nσ={sd:.4f}\n95%CI [{lo:.4f},{hi:.4f}]",
            ha="center", va="top", fontsize=7.5, color="#222222",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray", alpha=0.8))
plt.suptitle(f"PAD-UFES-20 {CONFIG} — Bootstrap Metric Distributions (n={N_BOOTSTRAP}) — Test Set",
             fontsize=13, fontweight="bold", y=0.97)
plt.savefig(f"/kaggle/working/pad_{CONFIG.lower()}_bootstrap_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

means  = np.array([np.mean(per_class_f1[n]) for n in class_names])
stds   = np.array([np.std(per_class_f1[n])  for n in class_names])
lo_arr = np.array([ci95(per_class_f1[n])[0] for n in class_names])
hi_arr = np.array([ci95(per_class_f1[n])[1] for n in class_names])

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(NUM_CLASSES)
bars = ax.bar(x, means, yerr=[means - lo_arr, hi_arr - means], capsize=6,
              color="#4c72b0", alpha=0.8, ecolor="#333333",
              error_kw={"linewidth": 1.8, "capthick": 1.8}, width=0.55)
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=11)
ax.set_ylabel("F1 Score", fontsize=11); ax.set_ylim(0, 1.15)
ax.set_title(f"PAD-UFES-20 {CONFIG} — Per-Class F1 Mean ± 95% CI (Bootstrap)",
             fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3, linestyle="--"); ax.spines[["top", "right"]].set_visible(False)
for bar, m, s, hi in zip(bars, means, stds, hi_arr):
    ax.text(bar.get_x() + bar.get_width() / 2, hi + 0.025,
            f"{m:.3f}\n±{s:.3f}", ha="center", va="bottom", fontsize=9, color="#111111")
plt.tight_layout()
plt.savefig(f"/kaggle/working/pad_{CONFIG.lower()}_bootstrap_perclass_f1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
cm = confusion_matrix(test_yt, test_yp)
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("PAD-UFES-20 Test Confusion Matrix — Config 3: Image + Metadata")
plt.tight_layout(); plt.show()